# ArkIve — Ajuste do corte do RAG

Varre `RAG_MIN_SCORE` e mede o custo de `RAG_MAX_CHARS` nos 108 casos rotulados de `rag/eval/casos.jsonl`, com o mesmo modelo int8 e o mesmo índice do Render. Não treina nada: roda em poucos minutos, em CPU.

## Como usar
1. Colab com runtime CPU (GPU não é necessária).
2. Em *Secrets*, cadastre `HF_TOKEN` (token **Read**; o repo `Sabelli/arkive-e5-vet` é privado) e ligue *Notebook access*.
3. *Executar tudo*. No fim, copie as linhas do `.env` e o JSON do baseline.

| Métrica | O que é |
|---|---|
| `recall@3_corte` | caso com doença esperada em que ela chega ao LLM (top-3 após o corte) |
| `ruido@3` | fração do top-3 que passa o corte sem ser relevante |
| `vazio_correto` | caso sem doença esperada em que nada passa o corte |

## 1. Instalação

Mesmas versões de `requirements.txt`: os scores saem iguais aos do Render.

In [ ]:
!pip -q install onnxruntime==1.30.0 tokenizers==0.23.2 huggingface_hub==1.33.0 faiss-cpu==1.15.1 python-dotenv

## 2. Repositório e modelo

O índice vem versionado em `rag/index/`; só o modelo e o tokenizer podado são baixados.

In [ ]:
import json, os, sys
from google.colab import userdata

REPO = "/content/arkive_clinical_engine"
if not os.path.exists(REPO):
    !git clone -q -b feature/rag https://github.com/2TDSPO-ArkIve/arkive_clinical_engine.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
sys.path.insert(0, REPO)

# secret do Colab tem prioridade sobre o .env (load_dotenv não sobrescreve)
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
except Exception as e:
    print(f"AVISO: secret HF_TOKEN indisponível ({type(e).__name__}); usando o do .env")
!python -m rag.retriever --download

from rag.eval import avaliar
from rag.retriever import RagRetriever, montar_contexto_rag

## 3. Busca

Top-10 de cada caso uma vez só; as células seguintes só recortam.

In [ ]:
rag = RagRetriever()
casos = [json.loads(l) for l in avaliar.CASOS.read_text(encoding="utf-8").splitlines() if l.strip()]
tops = [rag.buscar(c["relato"], c["especie"], k=10) for c in casos]
res = [[(d["titulo"], d["score"]) for d in t] for t in tops]
print(f"{len(casos)} casos | {len(rag.chunks)} chunks no índice")

## 4. Varredura de `RAG_MIN_SCORE`

Sugestão = menor limiar com `ruido@3` até `RUIDO_MAX`. Troque `MIN_SCORE` à mão se preferir outro ponto da tabela.

In [ ]:
RUIDO_MAX = 0.15
LIMIARES = [round(0.38 + i / 100, 2) for i in range(13)]  # 0.38 a 0.50

tabela = {}
print(f"{'limiar':>6} {'recall@3_corte':>14} {'ruido@3':>8} {'vazio_correto':>13} {'docs/caso':>9}")
for t in LIMIARES:
    m = tabela[t] = avaliar.metricas(casos, res, t)
    docs = sum(sum(s >= t for _, s in r[:3]) for r in res) / len(res)
    print(f"{t:>6.2f} {m['recall@3_corte']:>14.4f} {m['ruido@3']:>8.4f} {m['vazio_correto']:>13.4f} {docs:>9.2f}")

m0 = tabela[LIMIARES[0]]
print(f"\nsem corte: recall@3 {m0['recall@3']:.4f} (teto do recall@3_corte) | mrr@10 {m0['mrr@10']:.4f}")
cabem = [t for t, m in tabela.items() if m["ruido@3"] <= RUIDO_MAX]
MIN_SCORE = min(cabem) if cabem else LIMIARES[-1]
print(f"sugestão: RAG_MIN_SCORE={MIN_SCORE}")
if not cabem:
    print(f"AVISO: nenhum limiar fica com ruido@3 <= {RUIDO_MAX}; usando o maior da tabela")
elif MIN_SCORE == LIMIARES[0]:
    print("AVISO: o menor limiar já cabe no ruído; vale estender LIMIARES para baixo")

## 5. Custo de `RAG_MAX_CHARS`

Recuperação não mede isto: o orçamento só decide quanto de cada documento o LLM lê. A tabela usa os documentos que passam `MIN_SCORE` e mostra o bloco que iria ao prompt. Tokens ≈ chars / 4 (texto em inglês).

In [ ]:
def corpo(d):
    # mesmo texto que montar_contexto_rag corta
    return " […] ".join([d["resumo"]["texto"]] + [f"({c['secao']}) {c['texto']}" for c in d["trechos"]])

enviados = [e for e in ([d for d in t[:3] if d["score"] >= MIN_SCORE] for t in tops) if e]
print(f"{len(enviados)} de {len(tops)} casos mandam algum documento com RAG_MIN_SCORE={MIN_SCORE}\n")
print(f"{'max_chars':>9} {'chars/bloco':>11} {'~tokens':>7} {'% do doc lido':>13} {'docs inteiros':>13}")
for mc in (3000, 4500, 6000):
    chars = sum(len(montar_contexto_rag(e, mc)) for e in enviados) / len(enviados)
    # aproximado: ignora o cabeçalho de cada documento
    lido = [min(1.0, (mc // len(e)) / len(corpo(d))) for e in enviados for d in e]
    print(f"{mc:>9} {chars:>11.0f} {chars / 4:>7.0f} {100 * sum(lido) / len(lido):>12.0f}% "
          f"{100 * sum(x == 1.0 for x in lido) / len(lido):>12.0f}%")

## 6. Resultado

Cole as linhas no `.env` e no Render, e o JSON em `rag/eval/baseline.json`. Ajuste também o default de `RAG_MIN_SCORE` em `config.py`.

In [ ]:
m = {**tabela[MIN_SCORE], "min_score": MIN_SCORE}
print(f"RAG_MIN_SCORE={MIN_SCORE}")
print("RAG_MAX_CHARS=<escolha da tabela da seção 5>\n")
print("rag/eval/baseline.json:")
print(json.dumps(m, indent=2))